In [1]:
# Install required libraries
!pip install torch transformers scikit-learn pandas numpy tqdm

In [2]:
import json
import torch
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW

In [3]:
print("GPU Available:", torch.cuda.is_available())

GPU Available: True


In [4]:
with open("ESConv.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("Total Conversations:", len(data))

Total Conversations: 1300


In [5]:
texts = []
labels = []

for conversation in data:
    for turn in conversation["dialog"]:
        if turn["speaker"] == "supporter":
            if "strategy" in turn["annotation"]:
                strategy = turn["annotation"]["strategy"]
                text = turn["content"].strip()

                if strategy in ["Providing Suggestions", "Information", "Question"]:
                    texts.append(text)
                    labels.append(0)  # Directive Support

                elif strategy in ["Affirmation and Reassurance",
                                  "Reflection of feelings",
                                  "Restatement or Paraphrasing",
                                  "Self-disclosure"]:
                    texts.append(text)
                    labels.append(1)  # Emotional Validation

print("Total Samples:", len(texts))
print("Unique Labels:", set(labels))

Total Samples: 15035
Unique Labels: {0, 1}


In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Train: 10524
Validation: 2255
Test: 2256


In [7]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

max_length = 128

train_encodings = tokenizer(X_train, truncation=True, padding=True, max_length=max_length)
val_encodings = tokenizer(X_val, truncation=True, padding=True, max_length=max_length)
test_encodings = tokenizer(X_test, truncation=True, padding=True, max_length=max_length)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
class ESConvDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [9]:
train_dataset = ESConvDataset(train_encodings, y_train)
val_dataset = ESConvDataset(val_encodings, y_val)
test_dataset = ESConvDataset(test_encodings, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [10]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [11]:
optimizer = AdamW(model.parameters(), lr=2e-5)

In [12]:
epochs = 6

model.train()

for epoch in range(epochs):
    total_loss = 0

    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} - Loss: {total_loss/len(train_loader)}")

100%|██████████| 329/329 [03:40<00:00,  1.49it/s]


Epoch 1 - Loss: 0.4531369355583626


100%|██████████| 329/329 [03:52<00:00,  1.41it/s]


Epoch 2 - Loss: 0.350908723555075


100%|██████████| 329/329 [03:52<00:00,  1.41it/s]


...

Epoch 4 - Loss: 0.13514254762476882


100%|██████████| 329/329 [03:52<00:00,  1.41it/s]


Epoch 5 - Loss: 0.08626609498572657


100%|██████████| 329/329 [03:52<00:00,  1.41it/s]

Epoch 6 - Loss: 0.054984394825731704


In [13]:
model.eval()

preds = []
true_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(labels_batch.cpu().numpy())

accuracy = accuracy_score(true_labels, preds)

print("Test Accuracy:", accuracy)
print(classification_report(true_labels, preds,
      target_names=["Directive Support", "Emotional Validation"]))

100%|██████████| 71/71 [00:18<00:00,  3.90it/s]


Test Accuracy: 0.8111702127659575
                      precision    recall  f1-score   support

   Directive Support       0.83      0.81      0.82      1196
Emotional Validation       0.79      0.82      0.80      1060

            accuracy                           0.81      2256
           macro avg       0.81      0.81      0.81      2256
        weighted avg       0.81      0.81      0.81      2256

